# Evaluating Agentic RAG: Breaking and Fixing a Claims Assistant

In the first half of the lab, we used LangGraph to extract structured data from messy adjuster notes. Now, we flip the script: we are going to build a Retrieval-Augmented Generation (RAG) agent that answers adjuster questions using the **06-2025 FEMA Claims Manual**, and then we are going to rigorously evaluate its performance using Snowflake's native `CORTEX.EVALUATE` functions.

## Learning Objectives

1. Ingest and chunk a massive PDF manual into a **Cortex Search Service**
2. Build a baseline RAG prompt (and watch it fail on complex claims)
3. Upgrade to an Agentic workflow to improve retrieval accuracy
4. Programmatically measure groundedness, relevance, and hallucinations natively in Snowflake

---
# Section 1: Building the Knowledge Base

Before we can evaluate an agent, it needs something to read. The [06-2025 FEMA Claims Manual] is hundreds of pages of dense policy rules. 

**Where you are in the overall pipeline:**

```text
[ FEMA PDF ] → [ LangChain Splitter ] → [ Cortex Search Service ] → Agent
                       ↑ YOU ARE HERE

In [ ]:
### Cell 3 (Python Code)
```python
import json
import time
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.files import SnowflakeFile
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Connect to our Snowflake Session
session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
# 2. Extract and Chunk the PDF
# We use LangChain's recursive splitter to keep paragraphs together 
# while staying under the LLM context window limits.

# Note: In a true production environment, you might use Snowflake's native 
# CORTEX.PARSE_DOCUMENT or pdfplumber, but PyPDF/LangChain works great for this lab!
from langchain_community.document_loaders import PyPDFLoader

# We download the file from the stage to the notebook's ephemeral storage to parse it
stage_file_path = "@claims_extraction_repo/branches/main/manuals/06-2025-claims-manual.pdf"
local_file_path = "/tmp/06-2025-claims-manual.pdf"

session.file.get(stage_file_path, "/tmp/")

loader = PyPDFLoader(local_file_path)
pages = loader.load()
print(f"Loaded {len(pages)} pages from the Claims Manual.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} searchable chunks.")

# 3. Save chunks to a Snowflake Table
# We extract the text and metadata into a list of dictionaries, then convert to a Snowpark DataFrame
chunk_data = [
    {
        "CHUNK_ID": f"chunk_{i}", 
        "PAGE_NUM": chunk.metadata.get("page", 0),
        "CHUNK_TEXT": chunk.page_content
    } 
    for i, chunk in enumerate(chunks)
]

df_chunks = session.create_dataframe(chunk_data)
df_chunks.write.mode("overwrite").save_as_table("CLAIMS_MANUAL_CHUNKS")
print("Successfully saved chunks to CLAIMS_MANUAL_CHUNKS table.")

In [ ]:
# 4. Create the Cortex Search Service
# This one SQL command handles text embedding, vector database provisioning, and similarity search indexing!

create_css_sql = """
CREATE OR REPLACE CORTEX SEARCH SERVICE claims_manual_search_svc
  ON chunk_text
  ATTRIBUTES page_num
  WAREHOUSE = CLAIMS_LAB_WH
  TARGET_LAG = '1 minute'
  AS (
    SELECT chunk_id, page_num, chunk_text
    FROM CLAIMS_MANUAL_CHUNKS
  );
"""

session.sql(create_css_sql).collect()
print("Cortex Search Service 'claims_manual_search_svc' is live and indexing!")

> **Takeaway:** By defining `ATTRIBUTES page_num` in our Cortex Search Service, we are allowing our future Agent to pre-filter its vector searches (e.g., "Only search for this rule between pages 50 and 60"). This is a critical pattern for improving RAG accuracy on massive documents!

---
# Section 2: The "Naive" RAG Trap

Now that our FEMA Claims Manual is indexed in Cortex, let's build a basic RAG (Retrieval-Augmented Generation) pipeline. 

The standard approach most developers take on day one is:
1. Take the user's question.
2. Search the vector database for relevant chunks.
3. Stuff those chunks into a massive prompt.
4. Ask the LLM for an answer.

Let's see what happens when we apply this "Naive RAG" approach to a complicated flood claim. Flood insurance rules around **basements** and **enclosures** are notoriously strict—NFIP severely limits what contents are covered below the lowest elevated floor. 

Let's query our `FIMA_CLAIMS` table for a claim that might fall into this trap.

In [ ]:
# 1. Find a "Tricky" Claim 
# We are looking for a claim that has a basement (basement_type > 0) 
# and a high amount of contents damage claimed.

tricky_claim_df = session.sql("""
    SELECT 
        claim_id, 
        state,
        basement_type, 
        contents_damage_amount, 
        building_damage_amount,
        adjuster_notes
    FROM FIMA_CLAIMS
    WHERE basement_type IN ('1', '2', '3', '4') -- Codes for basements/enclosures
      AND contents_damage_amount > 15000
    LIMIT 1
""").collect()

claim = tricky_claim_df[0]

print(f"Target Claim ID: {claim['CLAIM_ID']}")
print(f"Basement Type: {claim['BASEMENT_TYPE']}")
print(f"Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']:,.2f}")
print(f"\nAdjuster Notes:\n{claim['ADJUSTER_NOTES']}")

### Querying the Cortex Search Service

Next, we will query our Cortex Search Service for rules regarding basement contents coverage. We'll use the `snowflake.core` Python API (the modern standard for interacting with Snowflake services natively) to pull the top 5 most relevant chunks from the manual.

In [ ]:
import pandas as pd
from snowflake.core import Root

# Initialize the Snowflake Core Root object
root = Root(session)
db = session.get_current_database()
sch = session.get_current_schema()

# Connect to our Search Service
search_svc = root.databases[db].schemas[sch].cortex_search_services["claims_manual_search_svc"]

# 2. Perform the Vector Search
search_query = "What personal property or contents are covered in a basement or crawlspace?"

# We ask for the top 5 chunks
search_results = search_svc.search(
    query=search_query,
    columns=["CHUNK_TEXT", "PAGE_NUM"],
    limit=5
)

# Extract the text chunks to use as context
retrieved_chunks = [row["CHUNK_TEXT"] for row in search_results.results]

print(f"Retrieved {len(retrieved_chunks)} chunks from the FEMA manual.")
print("\nPreview of top chunk:")
print("-" * 40)
print(retrieved_chunks[0][:300] + "...")

### The Naive RAG Prompt

Now we stuff everything—the claim details and the retrieved manual chunks—into a single prompt for `claude-3-5-sonnet`. We are asking the model to act as a claims advisor and determine if we should pay out the contents damage.

In [ ]:
# 3. Build the context string
context_str = "\n\n---\n\n".join(retrieved_chunks)

# 4. Construct the Prompt
system_prompt = """
You are an expert FEMA NFIP Flood Insurance Advisor. 
Review the provided claim details and use ONLY the provided FEMA Claims Manual excerpts to advise the adjuster.

Determine if the claimed contents damage should be paid out in full, partially, or denied based on the manual's rules regarding basements.
"""

user_prompt = f"""
CLAIM DETAILS:
- Claim ID: {claim['CLAIM_ID']}
- Basement Type Code: {claim['BASEMENT_TYPE']} (Has a basement/enclosure)
- Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']}
- Adjuster Notes: {claim['ADJUSTER_NOTES']}

FEMA MANUAL EXCERPTS:
{context_str}

Based on the manual excerpts, how should we handle the $15,000+ contents damage in the basement for this claim? Be definitive.
"""

# 5. Call Cortex Complete
rag_prompt = json.dumps([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
])

naive_response = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${rag_prompt}$$),
        {{'temperature': 0.1}}
    ) AS agent_response
""").collect()[0][0]

print("🤖 NAIVE RAG AGENT RESPONSE:")
print("=============================")
print(naive_response)

> **Takeaway:** Read the LLM's response carefully. Did it definitively list *exactly* what is covered (like washers, dryers, and freezers) and explicitly deny general personal property (like TVs and couches) which are excluded in basements under NFIP rules? 
>
> Often, a Naive RAG approach will confidently hallucinate or provide a vague "pay it" response because the basic text chunking cut the exclusion list in half, or the LLM lost the strict policy rule in the noise of the prompt. 

This is exactly why we cannot push AI into production without **Evaluation**. In the next section, we will use Snowflake's native evaluation tools to mathematically prove that this agent's response is lacking, and then we will build a better one.